In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# -----------------------------
# Ví dụ dữ liệu
# -----------------------------
data = {
    'age': [25, 32, 47, 51, 23, 28, 35, 60],
    'income': [50000, 80000, 120000, 90000, 45000, 70000, 110000, 150000],
    'city': ['Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Hà Nội', 'TP.HCM'],
    'education': ['Cao đẳng', 'Đại học', 'Thạc sĩ', 'Đại học', 'Cao đẳng', 'Thạc sĩ', 'Đại học', 'Tiến sĩ'],
    'gender': ['M', 'F', 'F', 'M', 'F', 'M', 'F', 'M'],
    'target': [0, 1, 1, 0, 0, 1, 1, 0]  # cột nhãn (nếu có)
}

df = pd.DataFrame(data)

# In dữ liệu gốc
print("Dữ liệu gốc:")
print(df)
print("\n")

# -----------------------------
# 1. Chỉ định các cột muốn giữ lại và xử lý
# -----------------------------
# Các cột bạn muốn giữ (loại bỏ những cột không cần thiết)
cols_to_keep = ['age', 'income', 'city', 'education', 'gender']

# Các cột numerical cần scale
numerical_cols = ['age', 'income']

# Các cột categorical cần one-hot
categorical_cols = ['city', 'education', 'gender']

# Kiểm tra các cột có tồn tại không
missing_cols = set(cols_to_keep) - set(df.columns)
if missing_cols:
    raise ValueError(f"Các cột sau không tồn tại trong dữ liệu: {missing_cols}")

df_filtered = df[cols_to_keep + ['target']]  # giữ lại target nếu có

X = df_filtered.drop(columns=['target'], errors='ignore')
y = df_filtered['target'] if 'target' in df_filtered.columns else None

# -----------------------------
# 2. Tạo ColumnTransformer
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),           # hoặc MinMaxScaler()
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols),
    ],
    remainder='drop'  # bỏ các cột không được chỉ định trong transformer
)

# -----------------------------
# 3. Tạo Pipeline (tùy chọn: thêm model ở cuối)
# -----------------------------
from sklearn.linear_model import LogisticRegression

# Nếu bạn muốn chỉ tiền xử lý
preprocess_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# Nếu bạn muốn thêm model (ví dụ Logistic Regression)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# -----------------------------
# 4. Fit và transform
# -----------------------------
# Chỉ tiền xử lý
X_preprocessed = preprocess_pipeline.fit_transform(X)

# Lấy tên cột sau one-hot
ohe = preprocessor.named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = numerical_cols + list(cat_feature_names)

X_preprocessed_df = pd.DataFrame(X_preprocessed, columns=feature_names)
print("Dữ liệu sau khi filter + one-hot + scale:")
print(X_preprocessed_df)
print("\nShape:", X_preprocessed_df.shape)

# -----------------------------
# 5. Nếu muốn train model
# -----------------------------
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    full_pipeline.fit(X_train, y_train)
    score = full_pipeline.score(X_test, y_test)
    print(f"\nĐộ chính xác trên tập test: {score:.3f}")

# -----------------------------
# 6. Cách lưu và tái sử dụng pipeline
# -----------------------------
import joblib
joblib.dump(preprocess_pipeline, 'preprocessor_pipeline.pkl')
# Để load lại: loaded_pipeline = joblib.load('preprocessor_pipeline.pkl')